# Baseline U-Net34 trên Kaggle — 1 GPU, train và đánh giá đầy đủ

Notebook tự chứa toàn bộ pipeline và đọc trực tiếp dataset `tinlngnguyn/skin-lesion-segmentation-hus` đã Attach tại `/kaggle/input`. Ảnh RGB được tiền xử lý bằng đúng thuật toán `simplest_cb(..., 1)` từ [`fredguth/skin-cancer/color_balance.py`](https://github.com/fredguth/skin-cancer/blob/master/color_balance.py), sau đó mới resize. Training chính thức được chia 3 fold; LR Finder chỉ dùng train subset, augmentation chỉ dùng cho train subset và checkpoint được chọn bằng Best Dice trên fold holdout sạch.

Sau khi cả 3 fold đã được khóa, notebook chỉ đánh giá từng fold model và equal-weight ensemble trên **official validation**. Kết quả ensemble theo từng ảnh được xếp hạng bằng Dice để in và lưu Top 10 tốt nhất/Top 10 tệ nhất. Official validation tuyệt đối không tham gia LR Finder, train hoặc chọn checkpoint.

> Trên Kaggle: `Add Input` dataset `tinlngnguyn/skin-lesion-segmentation-hus`, chọn Accelerator GPU (một GPU là đủ) và bật Internet nếu cache ResNet34 ImageNet chưa có. Artifact được ghi vào `/kaggle/working/unet34_kaggle_1gpu`.

In [ ]:
# Kaggle image đã có PyTorch, torchvision, OpenCV và Albumentations.
# Không pip install giữa session để tránh thay đổi binary/package đang chạy.
print('Using the Kaggle runtime environment.')

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from matplotlib.patches import Patch
from torch.utils.data import DataLoader, Dataset, get_worker_info
from torchvision import models
from tqdm.auto import tqdm

try:
    import psutil
except ImportError:
    psutil = None

# -------------------- Cấu hình baseline --------------------
DATASET_HANDLE = 'tinlngnguyn/skin-lesion-segmentation-hus'
DATA_ROOT = Path('/kaggle/input/skin-lesion-segmentation-hus')
SPLIT_DIRECTORIES = {
    'train': ('ISIC2018_Task1-2_Training_Input', 'ISIC2018_Task1_Training_GroundTruth'),
    'validation': ('ISIC2018_Task1-2_Validation_Input', 'ISIC2018_Task1_Validation_GroundTruth'),
}
EXPECTED_COUNTS = {'train': 2594, 'validation': 100}
SEED = 42
N_FOLDS = 3
SIZES = (128, 256)
FROZEN_EPOCHS = 30
UNFROZEN_EPOCHS = 30
PER_GPU_BATCH_BY_SIZE = {128: 16, 256: 8}
NUM_WORKERS = min(2, os.cpu_count() or 1)
PREFETCH_FACTOR = 1
PIN_MEMORY = True
CACHE_RESIZED_IN_RAM = True
DROP_OS_PAGE_CACHE = True
SAVE_LAST_EVERY = 5
LRF_ITERS = 50
LR_MIN, LR_MAX, LRF_BETA = 1e-6, 1.0, 0.9
LR_SELECTION_FACTOR = 0.1
MAX_LR_FROZEN, MAX_LR_UNFROZEN = 3e-3, 1e-3
CUT_FRAC, STLR_RATIO = 0.1, 32.0
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
PRED_THRESHOLD = 0.5
JACCARD_CUT = 0.65
PRETRAINED = True
USE_CHANNELS_LAST = False
FP32_TRANSPOSED_CONV = True
USE_FUSED_ADAM = False
RESUME = True
FAST_DEV_RUN = False
PROTOCOL_VERSION = 7
COLOR_BALANCE_PERCENT = 1.0
COLOR_BALANCE_SOURCE = 'fredguth/skin-cancer@master/color_balance.py'

if FAST_DEV_RUN:
    SIZES = (128,)
    FROZEN_EPOCHS = UNFROZEN_EPOCHS = 1
    LRF_ITERS = 8

if not hasattr(A, 'D4'):
    raise RuntimeError('Albumentations runtime quá cũ: cần A.D4 và A.Compose(seed=...).')
try:
    A.Compose([], seed=SEED)
except TypeError as error:
    raise RuntimeError('Albumentations runtime không hỗ trợ Compose(seed=...).') from error

OUTPUT_ROOT = Path('/kaggle/working/unet34_kaggle_1gpu')
RUN_NAME = ('fast_dev_v7_valid_only' if FAST_DEV_RUN
            else 'full_v7_valid_only')
RUN_DIR = OUTPUT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
LOG_DIR = RUN_DIR / 'logs'
for directory in (RUN_DIR, CHECKPOINT_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

set_seed()
cv2.setNumThreads(0)
torch.set_float32_matmul_precision('high')
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật Accelerator GPU trong Kaggle Settings.')

DEVICE = torch.device('cuda:0')
GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
BATCH_SIZE_BY_SIZE = dict(PER_GPU_BATCH_BY_SIZE)  # notebook luôn chỉ dùng cuda:0
USE_AMP = True
print({
    'device': str(DEVICE), 'gpus': GPU_NAMES, 'global_batch': BATCH_SIZE_BY_SIZE,
    'workers': NUM_WORKERS, 'channels_last': USE_CHANNELS_LAST,
    'fp32_upconv': FP32_TRANSPOSED_CONV, 'run_dir': str(RUN_DIR),
})

def memory_status():
    parts = []
    if psutil is not None:
        process = psutil.Process(os.getpid())
        info = process.memory_full_info()
        main_bytes = getattr(info, 'uss', info.rss)
        worker_bytes = 0
        for child in process.children(recursive=True):
            try:
                child_info = child.memory_full_info()
                worker_bytes += getattr(child_info, 'uss', child_info.rss)
            except psutil.Error:
                pass
        parts.extend([f'ram_main={main_bytes / 2**30:.2f}GiB',
                      f'ram_workers={worker_bytes / 2**30:.2f}GiB'])
    allocated = torch.cuda.memory_allocated(0)
    reserved = torch.cuda.memory_reserved(0)
    parts.extend([f'cuda_alloc={allocated / 2**30:.2f}GiB',
                  f'cuda_reserved={reserved / 2**30:.2f}GiB'])
    return ' '.join(parts)

In [ ]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

def find_populated_directory(directory_name, suffixes):
    outer = DATA_ROOT / directory_name
    candidates = [outer / directory_name, outer]
    if DATA_ROOT.is_dir():
        candidates.extend(path for path in DATA_ROOT.rglob(directory_name) if path.is_dir())
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen or not candidate.is_dir():
            continue
        seen.add(candidate)
        if any(path.is_file() and path.suffix.lower() in suffixes
               for path in candidate.iterdir()):
            return candidate
    raise FileNotFoundError(f'Không tìm thấy thư mục dữ liệu {directory_name} dưới {DATA_ROOT}')

def discover_records(split):
    image_name, mask_name = SPLIT_DIRECTORIES[split]
    image_dir = find_populated_directory(image_name, IMAGE_SUFFIXES)
    mask_dir = find_populated_directory(mask_name, {'.png'})
    image_paths = [path for path in image_dir.iterdir()
                   if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES]
    mask_paths = [path for path in mask_dir.iterdir()
                  if path.is_file() and path.suffix.lower() == '.png'
                  and path.stem.endswith('_segmentation')]
    if len({path.stem for path in image_paths}) != len(image_paths):
        raise RuntimeError(f'{split}: trùng image ID')
    image_map = {path.stem: path for path in image_paths}
    mask_map = {path.stem.removesuffix('_segmentation'): path for path in mask_paths}
    if len(mask_map) != len(mask_paths):
        raise RuntimeError(f'{split}: trùng mask ID')
    missing_masks = sorted(set(image_map) - set(mask_map))
    orphan_masks = sorted(set(mask_map) - set(image_map))
    if missing_masks or orphan_masks:
        raise RuntimeError(f'{split}: thiếu mask={missing_masks[:5]}, mask mồ côi={orphan_masks[:5]}')
    records = [
        {'image_id': image_id, 'image_path': image_map[image_id],
         'mask_path': mask_map[image_id]}
        for image_id in sorted(image_map)
    ]
    if len(records) != EXPECTED_COUNTS[split]:
        raise RuntimeError(f'{split}: cần {EXPECTED_COUNTS[split]} cặp, thấy {len(records)}')
    return records, image_dir, mask_dir

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f'Chưa Attach dataset {DATASET_HANDLE}; không thấy {DATA_ROOT}'
    )
train_records_all, train_image_dir, train_mask_dir = discover_records('train')
official_val_records_all, val_image_dir, val_mask_dir = discover_records('validation')

split_ids = {
    'train': {record['image_id'] for record in train_records_all},
    'validation': {record['image_id'] for record in official_val_records_all},
}
assert split_ids['train'].isdisjoint(split_ids['validation'])

def deterministic_subset(records, count, seed):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(records))[:min(count, len(records))]
    return [records[int(index)] for index in indices]

if FAST_DEV_RUN:
    train_records = deterministic_subset(train_records_all, 96, SEED)
    official_val_records = deterministic_subset(official_val_records_all, 24, SEED + 1)
else:
    train_records = train_records_all
    official_val_records = official_val_records_all

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_records), generator=generator).tolist()
fold_sizes = [len(train_records) // N_FOLDS] * N_FOLDS
for index in range(len(train_records) % N_FOLDS):
    fold_sizes[index] += 1
fold_indices, cursor = [], 0
for fold_size in fold_sizes:
    fold_indices.append(permutation[cursor:cursor + fold_size])
    cursor += fold_size
folds, all_indices = [], set(range(len(train_records)))
for holdout_indices in fold_indices:
    holdout_set = set(holdout_indices)
    train_indices = sorted(all_indices - holdout_set)
    train_subset = [train_records[index] for index in train_indices]
    holdout_subset = [train_records[index] for index in holdout_indices]
    assert {r['image_id'] for r in train_subset}.isdisjoint(
        r['image_id'] for r in holdout_subset
    )
    folds.append((train_subset, holdout_subset))

print('pairs:', {'train': len(train_records), 'validation': len(official_val_records)},
      'fold sizes:', fold_sizes)
print('train dirs:', train_image_dir, train_mask_dir, sep='\n  ')
print('validation dirs:', val_image_dir, val_mask_dir, sep='\n  ')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
RAM_CACHE = {}


def apply_mask(matrix, mask, fill_value):
    masked = np.ma.array(matrix, mask=mask, fill_value=fill_value)
    return masked.filled()


def apply_threshold(matrix, low_value, high_value):
    low_mask = matrix < low_value
    matrix = apply_mask(matrix, low_mask, low_value)

    high_mask = matrix > high_value
    matrix = apply_mask(matrix, high_mask, high_value)

    return matrix


def simplest_cb(img, percent):
    """Thuật toán từ fredguth/skin-cancer/color_balance.py."""
    assert img.shape[2] == 3
    assert percent > 0 and percent < 100

    half_percent = percent / 200.0
    channels = cv2.split(img)

    out_channels = []
    for channel in channels:
        assert len(channel.shape) == 2

        height, width = channel.shape
        vec_size = width * height
        flat = channel.reshape(vec_size)

        assert len(flat.shape) == 1

        flat = np.sort(flat)
        n_cols = flat.shape[0]

        low_val = flat[math.floor(n_cols * half_percent)]
        high_val = flat[math.ceil(n_cols * (1.0 - half_percent))]

        # Bản gốc print low/high ở đây để debug; bỏ print không đổi kết quả.
        thresholded = apply_threshold(channel, low_val, high_val)
        normalized = cv2.normalize(
            thresholded, thresholded.copy(), 0, 255, cv2.NORM_MINMAX
        )
        out_channels.append(normalized)

    return cv2.merge(out_channels)


def load_color_balanced_rgb(path):
    # color_balance.py dùng cv2.imread -> BGR. Giữ đúng thứ tự đó khi balance,
    # rồi mới đổi sang RGB vì encoder pretrained dùng ImageNet RGB.
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f'OpenCV không đọc được ảnh: {path}')
    if (DROP_OS_PAGE_CACHE and hasattr(os, 'posix_fadvise')
            and hasattr(os, 'POSIX_FADV_DONTNEED')):
        try:
            with open(path, 'rb') as handle:
                os.posix_fadvise(handle.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
        except OSError:
            pass
    balanced_bgr = simplest_cb(image_bgr, COLOR_BALANCE_PERCENT)
    balanced_rgb = cv2.cvtColor(balanced_bgr, cv2.COLOR_BGR2RGB)
    return balanced_rgb.astype(np.uint8, copy=False)


def load_pil_uint8(path, mode):
    with open(path, 'rb') as handle:
        with Image.open(handle) as source:
            array = np.asarray(source.convert(mode), dtype=np.uint8).copy()
        if (DROP_OS_PAGE_CACHE and hasattr(os, 'posix_fadvise')
                and hasattr(os, 'POSIX_FADV_DONTNEED')):
            try:
                os.posix_fadvise(handle.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
            except OSError:
                pass
    return array

def build_transform(size, train, seed, preprocessed=False):
    steps = []
    if not preprocessed:
        steps.extend([
            A.Resize(size, size, interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST),
        ])
    if train:
        steps.extend([
            A.D4(p=1.0),
            A.Affine(rotate=(-44, 44), scale=(1.0, 1.05),
                     interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST, p=0.75),
            A.RandomBrightnessContrast(brightness_limit=0.2,
                                       contrast_limit=0.2, p=0.75),
        ])
    steps.append(ToTensorV2())
    return A.Compose(steps, seed=int(seed))

class ISICDataset(Dataset):
    def __init__(self, records, size, train, seed=SEED):
        if not records:
            raise ValueError('Dataset rỗng')
        self.records = list(records)
        self.size = int(size)
        self.train = bool(train)
        cache = RAM_CACHE.get(self.size)
        indices = None if cache is None else [cache['id_to_index'].get(r['image_id'])
                                              for r in self.records]
        self.cache = cache if indices is not None and all(i is not None for i in indices) else None
        self.cache_indices = indices if self.cache is not None else None
        self.transform = build_transform(
            self.size, self.train, seed, preprocessed=self.cache is not None
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        if self.cache is not None:
            cache_index = self.cache_indices[index]
            image_np = self.cache['images'][cache_index].copy()
            mask_np = self.cache['masks'][cache_index].copy()
        else:
            image_np = load_color_balanced_rgb(record['image_path'])
            mask_np = load_pil_uint8(record['mask_path'], 'L')
            if image_np.shape[:2] != mask_np.shape:
                raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
            mask_np = (mask_np > 127).astype(np.uint8)
        transformed = self.transform(image=image_np, mask=mask_np)
        image, mask = transformed['image'], transformed['mask']
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        return image.contiguous(), (mask > 0).float().contiguous()

def seed_worker(worker_id):
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    worker = get_worker_info()
    if worker is not None and hasattr(worker.dataset.transform, 'set_random_seed'):
        worker.dataset.transform.set_random_seed(worker_seed)

def make_loader(records, size, train, seed, batch_size=None, drop_last=False):
    if batch_size is None:
        batch_size = BATCH_SIZE_BY_SIZE[int(size)]
    kwargs = dict(
        dataset=ISICDataset(records, size, train, seed),
        batch_size=int(batch_size), shuffle=bool(train), drop_last=bool(drop_last),
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=False,
        worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(int(seed)),
    )
    if NUM_WORKERS > 0:
        kwargs['prefetch_factor'] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

def build_ram_caches(records, sizes):
    records = list(records)
    sizes = tuple(dict.fromkeys(int(size) for size in sizes))
    image_ids = [record['image_id'] for record in records]
    if len(image_ids) != len(set(image_ids)):
        raise ValueError('Trùng image_id trong RAM cache')
    caches = {
        size: {
            'images': np.empty((len(records), size, size, 3), dtype=np.uint8),
            'masks': np.empty((len(records), size, size), dtype=np.uint8),
            'id_to_index': {image_id: index for index, image_id in enumerate(image_ids)},
        }
        for size in sizes
    }
    for index, record in enumerate(tqdm(records, desc=f'RAM cache {sizes}px')):
        image_np = load_color_balanced_rgb(record['image_path'])
        mask_np = load_pil_uint8(record['mask_path'], 'L')
        if image_np.shape[:2] != mask_np.shape:
            raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
        binary_mask = (mask_np > 127).astype(np.uint8)
        for size in sizes:
            resized = cv2.resize(image_np, (size, size), interpolation=cv2.INTER_LINEAR)
            caches[size]['images'][index] = resized
            caches[size]['masks'][index] = cv2.resize(
                binary_mask, (size, size), interpolation=cv2.INTER_NEAREST
            )
    return caches

if CACHE_RESIZED_IN_RAM:
    RAM_CACHE.update(build_ram_caches(train_records, SIZES))
    cache_gib = sum(v['images'].nbytes + v['masks'].nbytes
                    for v in RAM_CACHE.values()) / 2**30
    print(f'Train-only color-balanced + resized uint8 cache: {cache_gib:.2f} GiB | {memory_status()}')

clean_dataset = ISICDataset(train_records[:3], SIZES[0], train=False)
for index in range(len(clean_dataset)):
    x1, y1 = clean_dataset[index]
    x2, y2 = clean_dataset[index]
    assert torch.equal(x1, x2) and torch.equal(y1, y2)
    assert x1.dtype == torch.uint8 and y1.dtype == torch.float32
    assert set(torch.unique(y1).tolist()).issubset({0.0, 1.0})
    assert 0.0 < float(y1.mean()) < 1.0
del clean_dataset, x1, x2, y1, y2
print('Dataset, mask and clean-transform checks: OK')

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels // 2, 2, stride=2)
        self.skip_projection = nn.Conv2d(skip_channels, out_channels // 2, 1)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x, skip):
        source_dtype = x.dtype
        if FP32_TRANSPOSED_CONV and x.is_cuda:
            with torch.autocast(device_type='cuda', enabled=False):
                x = self.up(x.float())
            x = x.to(dtype=source_dtype)
        else:
            x = self.up(x)
        skip = self.skip_projection(skip)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='nearest')
        return self.bn(F.relu(torch.cat([x, skip], dim=1)))

class UNet34(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        try:
            backbone = models.resnet34(weights=weights)
        except Exception as error:
            raise RuntimeError(
                'Không tải được ResNet34 ImageNet weights; hãy bật Internet trong Kaggle.'
            ) from error
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 256)
        self.up2 = UpBlock(256, 64, 256)
        self.up1 = UpBlock(256, 64, 256)
        self.up0 = UpBlock(256, 3, 16)
        self.head = nn.Conv2d(16, 1, 1)

    @property
    def first_layer_group(self):
        return (self.stem, self.layer1, self.layer2)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.layer1(self.pool(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        bottleneck = self.layer4(s3)
        decoded = self.up4(bottleneck, s3)
        decoded = self.up3(decoded, s2)
        decoded = self.up2(decoded, s1)
        decoded = self.up1(decoded, s0)
        decoded = self.up0(decoded, x)
        return self.head(decoded)

def unwrap_model(model):
    return model

def to_runtime(model):
    model = model.to(DEVICE)
    if USE_CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    return model  # intentionally use only cuda:0

def freeze_first_layer_group(model):
    base = unwrap_model(model)
    for parameter in base.parameters():
        parameter.requires_grad = True
    for module in base.first_layer_group:
        for child in module.modules():
            for parameter in child.parameters(recurse=False):
                parameter.requires_grad = isinstance(child, nn.BatchNorm2d)

def unfreeze_with_frozen_bn(model):
    for parameter in model.parameters():
        parameter.requires_grad = True
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            for parameter in module.parameters():
                parameter.requires_grad = False

def set_train_mode(model, freeze_bn):
    model.train()
    if freeze_bn:
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

def cpu_state_dict(model):
    return {key: value.detach().cpu().clone()
            for key, value in unwrap_model(model).state_dict().items()}

@torch.no_grad()
def refresh_cpu_state_dict(model, destination):
    source = unwrap_model(model).state_dict()
    if tuple(source) != tuple(destination):
        raise RuntimeError('state_dict keys changed during training')
    for key, value in source.items():
        destination[key].copy_(value.detach(), non_blocking=False)

def load_model_state(model, state):
    while state and all(key.startswith('module.') for key in state):
        state = {key.removeprefix('module.'): value for key, value in state.items()}
    unwrap_model(model).load_state_dict(state)

shape_model = UNet34(pretrained=False).eval()
with torch.inference_mode():
    shape_output = shape_model(torch.randn(2, 3, 128, 128))
assert shape_output.shape == (2, 1, 128, 128)
assert not any(key.startswith('first_layer_group.') for key in shape_model.state_dict())
print(f'Parameters: {sum(p.numel() for p in shape_model.parameters()) / 1e6:.3f}M')
del shape_model, shape_output

In [ ]:
criterion = nn.BCEWithLogitsLoss()
INPUT_DTYPE = torch.float16 if USE_AMP else torch.float32
GPU_MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE, dtype=INPUT_DTYPE).view(1, 3, 1, 1)
GPU_STD = torch.tensor(IMAGENET_STD, device=DEVICE, dtype=INPUT_DTYPE).view(1, 3, 1, 1)

def prepare_inputs(inputs):
    inputs = inputs.to(DEVICE, non_blocking=True).to(dtype=INPUT_DTYPE)
    inputs.div_(255.0).sub_(GPU_MEAN).div_(GPU_STD)
    if USE_CHANNELS_LAST:
        return inputs.contiguous(memory_format=torch.channels_last)
    return inputs.contiguous()

def per_image_dice(probability, target, threshold=PRED_THRESHOLD):
    prediction = (probability > threshold).float()
    dims = (1, 2, 3)
    intersection = (prediction * target).sum(dims)
    denominator = prediction.sum(dims) + target.sum(dims)
    return torch.where(denominator > 0, 2 * intersection / denominator.clamp(min=1),
                       torch.ones_like(denominator))

@torch.inference_mode()
def evaluate_holdout_metrics(model, loader):
    model.eval()
    metric_sums = torch.zeros(2, device=DEVICE, dtype=torch.float32)
    sample_count = 0
    for inputs, targets in loader:
        inputs = prepare_inputs(inputs)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            probability = torch.sigmoid(model(inputs).float())
        if not bool(torch.isfinite(probability).all()):
            raise FloatingPointError('Model sinh xác suất NaN/Inf')
        prediction = (probability > PRED_THRESHOLD).float()
        dims = (1, 2, 3)
        intersection = (prediction * targets).sum(dims)
        prediction_sum = prediction.sum(dims)
        target_sum = targets.sum(dims)
        denominator = prediction_sum + target_sum
        dice = torch.where(
            denominator > 0,
            2 * intersection / denominator.clamp(min=1),
            torch.ones_like(denominator),
        )
        union = prediction_sum + target_sum - intersection
        jaccard = torch.where(
            union > 0, intersection / union.clamp(min=1), torch.ones_like(union)
        )
        metric_sums[0] += dice.sum()
        metric_sums[1] += (jaccard * (jaccard >= JACCARD_CUT)).sum()
        sample_count += targets.shape[0]
    if sample_count == 0:
        raise RuntimeError('Holdout loader rỗng')
    dice, threshold_jaccard = (metric_sums / sample_count).cpu().tolist()
    return {
        'dice': float(dice),
        'threshold_jaccard': float(threshold_jaccard),
    }

def evaluate_dice(model, loader):
    return evaluate_holdout_metrics(model, loader)['dice']

def per_image_scores(probability, target, threshold=PRED_THRESHOLD):
    prediction = (probability > threshold).float()
    dims = (1, 2, 3)
    intersection = (prediction * target).sum(dims)
    prediction_sum = prediction.sum(dims)
    target_sum = target.sum(dims)
    union = prediction_sum + target_sum - intersection
    jaccard = torch.where(union > 0, intersection / union.clamp(min=1),
                           torch.ones_like(union))
    dice = torch.where(prediction_sum + target_sum > 0,
                       2 * intersection / (prediction_sum + target_sum).clamp(min=1),
                       torch.ones_like(union))
    sensitivity = torch.where(target_sum > 0,
                              intersection / target_sum.clamp(min=1),
                              torch.ones_like(target_sum))
    return jaccard, dice, sensitivity

# -------------------- T4 GPU evaluation backend --------------------
# Inference, Dice/IoU/Sensitivity and HD95 stay on CUDA.
# HD95 uses surface-to-surface Euclidean distance on GPU. The source/target
# boundary points are chunked so torch.cdist does not create a huge matrix.
EVAL_BATCH_SIZE = 16
HD95_SOURCE_CHUNK = 1024
HD95_TARGET_CHUNK = 4096

HD95_CROSS_KERNEL = torch.tensor(
    [[0.0, 1.0, 0.0],
     [1.0, 1.0, 1.0],
     [0.0, 1.0, 0.0]],
    device=DEVICE, dtype=torch.float32,
).view(1, 1, 3, 3)


def mask_surface_batch_gpu(masks):
    """Equivalent to scipy.ndimage.binary_erosion(mask) XOR mask for 2-D masks."""
    masks = masks.bool()
    neighborhood_count = F.conv2d(
        masks[:, None].float(), HD95_CROSS_KERNEL, padding=1
    )[:, 0]
    eroded = masks & (neighborhood_count == 5.0)
    return masks & ~eroded


def min_surface_distance_gpu(source_coords, target_coords):
    """Nearest Euclidean distance for every source point, memory-bounded on CUDA."""
    source_coords = source_coords.to(dtype=torch.float32)
    target_coords = target_coords.to(dtype=torch.float32)
    outputs = []
    for source_chunk in source_coords.split(HD95_SOURCE_CHUNK):
        chunk_min = torch.full(
            (source_chunk.shape[0],), float('inf'),
            device=source_chunk.device, dtype=torch.float32,
        )
        for target_chunk in target_coords.split(HD95_TARGET_CHUNK):
            distances = torch.cdist(source_chunk, target_chunk, p=2)
            chunk_min = torch.minimum(chunk_min, distances.amin(dim=1))
        outputs.append(chunk_min)
    return torch.cat(outputs, dim=0)


def hd95_batch_sum_gpu(prediction_masks, target_masks,
                       target_surfaces=None, target_nonempty=None):
    """
    Sum HD95 over a batch on GPU.
    Definition matches the old CPU implementation:
    95th percentile of bidirectional surface-to-surface distances.
    """
    prediction_masks = prediction_masks.bool()
    target_masks = target_masks.bool()

    if target_surfaces is None:
        target_surfaces = mask_surface_batch_gpu(target_masks)
    prediction_surfaces = mask_surface_batch_gpu(prediction_masks)

    prediction_nonempty = prediction_masks.flatten(1).any(dim=1).cpu().tolist()
    if target_nonempty is None:
        target_nonempty = target_masks.flatten(1).any(dim=1).cpu().tolist()

    height, width = target_masks.shape[-2:]
    empty_penalty = float(math.hypot(height, width))
    total = torch.zeros((), device=DEVICE, dtype=torch.float32)

    for batch_index, (pred_has_pixels, target_has_pixels) in enumerate(
        zip(prediction_nonempty, target_nonempty)
    ):
        if not pred_has_pixels and not target_has_pixels:
            continue
        if not pred_has_pixels or not target_has_pixels:
            total += empty_penalty
            continue

        prediction_coords = prediction_surfaces[batch_index].nonzero(as_tuple=False)
        target_coords = target_surfaces[batch_index].nonzero(as_tuple=False)

        pred_to_target = min_surface_distance_gpu(prediction_coords, target_coords)
        target_to_pred = min_surface_distance_gpu(target_coords, prediction_coords)
        distances = torch.cat((pred_to_target, target_to_pred), dim=0)
        total += torch.quantile(distances, 0.95)

    return total


@torch.inference_mode()
def model_probability_gpu(model, inputs):
    model.eval()
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
        return torch.sigmoid(model(inputs).float())


def accumulate_variant_gpu(metric_sum, probability, targets_device,
                           target_masks, target_surfaces, target_nonempty,
                           include_hd95):
    jaccard, dice, sensitivity = per_image_scores(probability, targets_device)
    metric_sum[0] += jaccard.sum()
    metric_sum[1] += (jaccard * (jaccard >= JACCARD_CUT)).sum()
    metric_sum[2] += dice.sum()
    metric_sum[3] += sensitivity.sum()

    if include_hd95:
        prediction_masks = probability[:, 0] > PRED_THRESHOLD
        metric_sum[4] += hd95_batch_sum_gpu(
            prediction_masks,
            target_masks,
            target_surfaces=target_surfaces,
            target_nonempty=target_nonempty,
        )
    return jaccard, dice, sensitivity


@torch.inference_mode()
def evaluate_variants_gpu(models_list, loader, individual_indices=(),
                          include_ensemble=True, include_hd95=True,
                          sample_ids=None):
    """
    Evaluate requested single models + equal-weight ensemble in ONE loader pass.
    Each model is forwarded only once per batch; probabilities are reused.
    """
    if not models_list:
        raise ValueError('models_list rỗng')
    if sample_ids is not None and not include_ensemble:
        raise ValueError('sample_ids yêu cầu include_ensemble=True')
    sample_ids = None if sample_ids is None else list(sample_ids)

    individual_indices = tuple(dict.fromkeys(int(i) for i in individual_indices))
    if any(i < 0 or i >= len(models_list) for i in individual_indices):
        raise IndexError('individual_indices ngoài phạm vi models_list')

    variant_keys = [f'model_{i}' for i in individual_indices]
    if include_ensemble:
        variant_keys.append('ensemble')

    metric_sums = {
        key: torch.zeros(5, device=DEVICE, dtype=torch.float32)
        for key in variant_keys
    }
    sample_count = 0
    per_image_rows = []
    detail_cursor = 0

    for inputs, targets in loader:
        inputs = prepare_inputs(inputs)
        targets_device = targets.to(DEVICE, non_blocking=True)
        target_masks = targets_device[:, 0] > 0.5

        target_surfaces = None
        target_nonempty = None
        if include_hd95:
            target_surfaces = mask_surface_batch_gpu(target_masks)
            target_nonempty = target_masks.flatten(1).any(dim=1).cpu().tolist()

        probabilities = []
        finite_flags = []
        for model in models_list:
            probability = model_probability_gpu(model, inputs)
            probabilities.append(probability)
            finite_flags.append(torch.isfinite(probability).all())

        if not bool(torch.stack(finite_flags).all()):
            raise FloatingPointError('Model sinh xác suất NaN/Inf khi evaluation')

        for model_index in individual_indices:
            accumulate_variant_gpu(
                metric_sums[f'model_{model_index}'],
                probabilities[model_index],
                targets_device,
                target_masks,
                target_surfaces,
                target_nonempty,
                include_hd95,
            )

        if include_ensemble:
            ensemble_probability = torch.zeros_like(probabilities[0])
            for probability in probabilities:
                ensemble_probability.add_(probability)
            ensemble_probability.div_(len(probabilities))
            jaccard, dice, sensitivity = accumulate_variant_gpu(
                metric_sums['ensemble'],
                ensemble_probability,
                targets_device,
                target_masks,
                target_surfaces,
                target_nonempty,
                include_hd95,
            )
            if sample_ids is not None:
                batch_count = targets_device.shape[0]
                batch_ids = sample_ids[detail_cursor:detail_cursor + batch_count]
                if len(batch_ids) != batch_count:
                    raise RuntimeError('Số sample_id ngắn hơn validation loader')
                predicted_pixels = (
                    ensemble_probability > PRED_THRESHOLD
                ).flatten(1).sum(dim=1)
                target_pixels = target_masks.flatten(1).sum(dim=1)
                values = torch.stack((
                    jaccard, dice, sensitivity,
                    predicted_pixels.float(), target_pixels.float(),
                ), dim=1).detach().cpu().tolist()
                for image_id, row_values in zip(batch_ids, values):
                    image_jaccard, image_dice, image_sensitivity, pred_px, target_px = row_values
                    per_image_rows.append({
                        'image_id': image_id,
                        'jaccard': float(image_jaccard),
                        'threshold_jaccard': float(
                            image_jaccard if image_jaccard >= JACCARD_CUT else 0.0
                        ),
                        'dice': float(image_dice),
                        'sensitivity': float(image_sensitivity),
                        'predicted_pixels': int(pred_px),
                        'target_pixels': int(target_px),
                    })
                detail_cursor += batch_count

        sample_count += targets_device.shape[0]

    if sample_count == 0:
        raise RuntimeError('Evaluation loader rỗng')
    if sample_ids is not None and detail_cursor != len(sample_ids):
        raise RuntimeError(
            f'Validation loader trả {detail_cursor} mẫu nhưng có {len(sample_ids)} ID'
        )

    # Only one GPU -> CPU synchronization per final result vector.
    results = {}
    names = ('jaccard', 'threshold_jaccard', 'dice', 'sensitivity', 'hd95')
    for key, metric_sum in metric_sums.items():
        values = (metric_sum / sample_count).detach().cpu().tolist()
        result = {
            'jaccard': float(values[0]),
            'threshold_jaccard': float(values[1]),
            'dice': float(values[2]),
            'sensitivity': float(values[3]),
        }
        if include_hd95:
            result['hd95'] = float(values[4])
        results[key] = result
    if sample_ids is not None:
        return results, per_image_rows
    return results


metric_target = torch.zeros(2, 1, 8, 8, device=DEVICE)
metric_target[0, :, :4] = 1
metric_target[1, :, :, :4] = 1
metric_probability = metric_target.clone()
metric_probability[1] = 0
metric_probability[1, :, :, :2] = 1
assert torch.allclose(per_image_dice(metric_probability, metric_target),
                      torch.tensor([1.0, 2 / 3], device=DEVICE), atol=1e-6)
del metric_target, metric_probability
print('Dice metric check: OK')

In [ ]:
def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

def trusted_torch_load(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def trainable_parameters(model):
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not parameters:
        raise RuntimeError('Không có parameter trainable')
    return parameters

def optimizer_for(model, learning_rate):
    parameters = trainable_parameters(model)
    if USE_FUSED_ADAM:
        try:
            return torch.optim.Adam(parameters, lr=learning_rate,
                                    weight_decay=WEIGHT_DECAY, fused=True)
        except (TypeError, RuntimeError):
            pass
    return torch.optim.Adam(parameters, lr=learning_rate, weight_decay=WEIGHT_DECAY)

def train_batch(model, optimizer, scaler, inputs, targets):
    inputs = prepare_inputs(inputs)
    targets = targets.to(DEVICE, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
        loss = criterion(model(inputs).float(), targets)
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError('Train loss là NaN/Inf')
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(
        (p for group in optimizer.param_groups for p in group['params']),
        GRAD_CLIP_NORM,
    )
    scaler.step(optimizer)
    scaler.update()
    return loss.detach()

def lr_find(model, records, size, stage_seed, freeze_bn):
    set_seed(stage_seed)
    original_state = cpu_state_dict(model)
    optimizer = optimizer_for(model, LR_MIN)
    scaler = make_scaler()
    loader = make_loader(records, size, train=True, seed=stage_seed,
                         batch_size=BATCH_SIZE_BY_SIZE[size], drop_last=False)
    iterator = iter(loader)
    multiplier = (LR_MAX / LR_MIN) ** (1 / max(1, LRF_ITERS - 1))
    learning_rates, losses = [], []
    moving_average, best_loss = 0.0, float('inf')
    for iteration in range(LRF_ITERS):
        try:
            inputs, targets = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            inputs, targets = next(iterator)
        learning_rate = LR_MIN * multiplier ** iteration
        for group in optimizer.param_groups:
            group['lr'] = learning_rate
        set_train_mode(model, freeze_bn)
        try:
            raw_loss = float(train_batch(model, optimizer, scaler, inputs, targets).cpu())
        except FloatingPointError:
            break
        moving_average = LRF_BETA * moving_average + (1 - LRF_BETA) * raw_loss
        smoothed = moving_average / (1 - LRF_BETA ** (iteration + 1))
        if not np.isfinite(smoothed) or (iteration > 5 and smoothed > 4 * best_loss):
            break
        best_loss = min(best_loss, smoothed)
        learning_rates.append(learning_rate)
        losses.append(smoothed)
    del iterator, loader, optimizer, scaler, inputs, targets
    load_model_state(model, original_state)
    del original_state
    gc.collect()
    set_seed(stage_seed)
    learning_rates = np.asarray(learning_rates, dtype=float)
    losses = np.asarray(losses, dtype=float)
    if len(learning_rates) < 3:
        raise RuntimeError('LR Finder không đủ điểm hợp lệ')
    gradient = np.gradient(losses, np.log10(learning_rates))
    steepest_lr = float(learning_rates[int(np.argmin(gradient))])
    min_loss_lr = float(learning_rates[int(np.argmin(losses))])
    cap = MAX_LR_UNFROZEN if freeze_bn else MAX_LR_FROZEN
    selected_lr = max(LR_MIN, min(steepest_lr * LR_SELECTION_FACTOR,
                                  min_loss_lr / 10, cap))
    print(f'LR Finder {size}px: selected={selected_lr:.2e}')
    return selected_lr, learning_rates.tolist(), losses.tolist()

def stlr_lambda(total_steps, cut_frac=CUT_FRAC, ratio=STLR_RATIO):
    cut = max(1, int(total_steps * cut_frac))
    def schedule(step):
        if step < cut:
            progress = step / cut
        else:
            denominator = cut * (1 / cut_frac - 1)
            progress = max(0.0, 1 - (step - cut) / max(1, denominator))
        return (1 + progress * (ratio - 1)) / ratio
    return schedule

def ids_digest(records):
    text = '\n'.join(record['image_id'] for record in records)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def config_digest(config):
    canonical = json.dumps(config, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest()

def save_best_state(best_state, best_path, metadata):
    atomic_torch_save({'model_state': best_state, 'meta': metadata}, best_path)

def run_stage(model, train_subset, holdout_subset, size, epochs, freeze_bn,
              fold_number, stage_name, stage_seed):
    stage_dir = CHECKPOINT_DIR / f'fold_{fold_number}' / f'{size}px_{stage_name}'
    best_path = stage_dir / 'best.pt'
    last_path = stage_dir / 'last.pt'
    contract = {
        'protocol_version': PROTOCOL_VERSION, 'fold': fold_number, 'size': size,
        'stage': stage_name, 'epochs': epochs, 'stage_seed': stage_seed,
        'freeze_bn': freeze_bn, 'batch_size': BATCH_SIZE_BY_SIZE[size],
        'train_ids': ids_digest(train_subset), 'holdout_ids': ids_digest(holdout_subset),
        'selection_metric': 'internal_fold_holdout_dice',
        'dataset_handle': DATASET_HANDLE, 'amp': USE_AMP,
        'channels_last': USE_CHANNELS_LAST,
        'fp32_transposed_conv': FP32_TRANSPOSED_CONV,
        'run_config_sha256': config_digest(RUN_CONFIG),
    }

    saved = None
    if RESUME and last_path.is_file():
        saved = trusted_torch_load(last_path, map_location='cpu')
        if saved.get('contract') != contract:
            raise RuntimeError(f'Checkpoint không khớp config/split: {last_path}')
        if saved.get('complete'):
            best_payload = trusted_torch_load(best_path, map_location='cpu')
            if best_payload.get('meta', {}).get('contract') != contract:
                raise RuntimeError(f'Best checkpoint không khớp: {best_path}')
            load_model_state(model, best_payload['model_state'])
            print(f'[resume] fold={fold_number} {size}px {stage_name}: complete')
            return saved['result']

    holdout_loader = make_loader(holdout_subset, size, train=False, seed=stage_seed)
    if saved is None:
        selected_lr, lrf_lrs, lrf_losses = lr_find(
            model, train_subset, size, stage_seed, freeze_bn
        )
        initial_metrics = evaluate_holdout_metrics(model, holdout_loader)
        initial_dice = initial_metrics['dice']
        best_score, best_epoch = initial_dice, 0
        history = []
        best_state = cpu_state_dict(model)
        best_meta = {
            'epoch': best_epoch, 'dice': best_score, 'selected_lr': selected_lr,
            'contract': contract,
        }
        save_best_state(best_state, best_path, best_meta)
        start_epoch = 0
    else:
        selected_lr = float(saved['selected_lr'])
        lrf_lrs, lrf_losses = saved['lrf_lrs'], saved['lrf_losses']
        best_score = float(saved['best_score'])
        best_epoch = int(saved['best_epoch'])
        history = list(saved['history'])
        best_payload = trusted_torch_load(best_path, map_location='cpu')
        best_meta = best_payload.get('meta', {})
        if best_meta.get('contract') != contract:
            raise RuntimeError(f'Best checkpoint không khớp: {best_path}')
        if int(best_meta['epoch']) >= best_epoch:
            best_epoch = int(best_meta['epoch'])
            best_score = float(best_meta['dice'])
        best_state = best_payload['model_state']
        del best_payload
        start_epoch = int(saved['next_epoch'])
        load_model_state(model, saved['model_state'])
        print(f'[resume] fold={fold_number} {size}px {stage_name}: epoch {start_epoch + 1}')

    steps_per_epoch = math.ceil(len(train_subset) / BATCH_SIZE_BY_SIZE[size])
    optimizer = optimizer_for(model, selected_lr)
    scaler = make_scaler()
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, stlr_lambda(epochs * steps_per_epoch)
    )
    if saved is not None:
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        scaler.load_state_dict(saved['scaler_state'])
        saved = None
        gc.collect()

    for epoch in range(start_epoch, epochs):
        epoch_started = time.perf_counter()
        train_loader = make_loader(train_subset, size, train=True,
                                   seed=stage_seed + epoch, drop_last=False)
        set_train_mode(model, freeze_bn)
        loss_sum = torch.zeros((), device=DEVICE)
        samples_seen = 0
        for inputs, targets in train_loader:
            loss_sum += train_batch(model, optimizer, scaler, inputs, targets)
            samples_seen += inputs.shape[0]
            scheduler.step()
        del inputs, targets
        train_loss = float((loss_sum / steps_per_epoch).cpu())
        del train_loader
        holdout_metrics = evaluate_holdout_metrics(model, holdout_loader)
        holdout_dice = holdout_metrics['dice']
        holdout_threshold_jaccard = holdout_metrics['threshold_jaccard']
        row = {
            'fold': fold_number, 'size': size, 'stage': stage_name,
            'epoch': epoch + 1, 'train_loss': train_loss,
            'holdout_dice': holdout_dice,
            'holdout_threshold_jaccard': holdout_threshold_jaccard,
            'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)
        if holdout_dice > best_score:
            best_score, best_epoch = holdout_dice, epoch + 1
            refresh_cpu_state_dict(model, best_state)
        if (epoch + 1) % SAVE_LAST_EVERY == 0 or epoch + 1 == epochs:
            best_meta = {
                'epoch': best_epoch, 'dice': best_score, 'selected_lr': selected_lr,
                'contract': contract,
            }
            save_best_state(best_state, best_path, best_meta)
            progress = {
                'complete': False, 'contract': contract, 'next_epoch': epoch + 1,
                'selected_lr': selected_lr, 'lrf_lrs': lrf_lrs,
                'lrf_losses': lrf_losses, 'best_score': best_score,
                'best_epoch': best_epoch, 'history': history,
                'model_state': unwrap_model(model).state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
            }
            atomic_torch_save(progress, last_path)
            del progress
        seconds = time.perf_counter() - epoch_started
        print(
            f'[fold {fold_number} | {size}px | {stage_name}] '
            f'epoch {epoch + 1:02d}/{epochs} loss={train_loss:.4f} '
            f'holdout_dice={holdout_dice:.4f} '
            f'threshold_jaccard={holdout_threshold_jaccard:.4f} '
            f'best={best_score:.4f} '
            f'time={seconds:.1f}s img/s={samples_seen / seconds:.1f} {memory_status()}'
        )

    load_model_state(model, best_state)
    result = {
        'fold': fold_number, 'size': size, 'stage': stage_name,
        'selected_lr': selected_lr, 'best_epoch': best_epoch,
        'best_holdout_dice': best_score, 'history': history,
        'lrf_lrs': lrf_lrs, 'lrf_losses': lrf_losses,
        'best_path': str(best_path),
    }
    atomic_torch_save({'complete': True, 'contract': contract, 'result': result}, last_path)
    del best_state, holdout_loader, optimizer, scaler, scheduler
    gc.collect()
    return result

In [ ]:
def preflight_batch(size):
    batch_size = BATCH_SIZE_BY_SIZE[size]
    model = to_runtime(UNet34(pretrained=False))
    optimizer = optimizer_for(model, 1e-4)
    scaler = make_scaler()
    inputs = torch.randint(0, 256, (batch_size, 3, size, size), dtype=torch.uint8)
    targets = torch.randint(0, 2, (batch_size, 1, size, size), dtype=torch.float32)
    try:
        loss = float(train_batch(model, optimizer, scaler, inputs, targets).cpu())
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            f'Batch {batch_size} không vừa GPU ở {size}px; giảm PER_GPU_BATCH_BY_SIZE.'
        ) from error
    finally:
        del model, optimizer, scaler, inputs, targets
        gc.collect()
        torch.cuda.empty_cache()
    print(f'Preflight {size}px batch={batch_size}: loss={loss:.4f}')

for size in SIZES:
    preflight_batch(size)

In [ ]:
RUN_CONFIG = {
    'dataset_handle': DATASET_HANDLE,
    'expected_counts': EXPECTED_COUNTS,
    'official_splits_used': ['Training', 'Validation'],
    'evaluation_scope': 'official_validation_only',
    'official_validation_evaluated': True, 'official_test_evaluated': False,
    'per_image_ranking': 'equal_weight_ensemble_dice_top10_best_and_worst',
    'seed': SEED, 'n_folds': N_FOLDS, 'sizes': list(SIZES),
    'frozen_epochs': FROZEN_EPOCHS, 'unfrozen_epochs': UNFROZEN_EPOCHS,
    'batch_size_by_size': BATCH_SIZE_BY_SIZE, 'num_workers': NUM_WORKERS,
    'prefetch_factor': PREFETCH_FACTOR, 'pin_memory': PIN_MEMORY,
    'pretrained': PRETRAINED, 'selection_metric': 'internal_fold_holdout_dice',
    'loss': 'BCEWithLogitsLoss',
    'optimizer': 'Adam(fused)' if USE_FUSED_ADAM else 'Adam',
    'weight_decay': WEIGHT_DECAY, 'grad_clip_norm': GRAD_CLIP_NORM,
    'lr_finder': {'min': LR_MIN, 'max': LR_MAX, 'iterations': LRF_ITERS,
                  'beta': LRF_BETA, 'selection_factor': LR_SELECTION_FACTOR,
                  'max_lr_frozen': MAX_LR_FROZEN,
                  'max_lr_unfrozen': MAX_LR_UNFROZEN},
    'stlr': {'cut_frac': CUT_FRAC, 'ratio': STLR_RATIO},
    'augmentation': 'D4+Affine(rotate44,scale1.0-1.05)+BrightnessContrast0.2',
    'augmentation_only_on_train_subset': True,
    'preprocessing': 'opencv_bgr+upstream_simplest_cb_1pct+bgr_to_rgb+resize+imagenet_normalize_on_gpu',
    'color_balance': {
        'method': 'simplest_cb', 'percent': COLOR_BALANCE_PERCENT,
        'tail_percent_per_side': COLOR_BALANCE_PERCENT / 2,
        'source': COLOR_BALANCE_SOURCE,
        'decode': 'cv2.imread BGR', 'postprocess': 'cv2 BGR-to-RGB',
        'applied_before_resize': True,
    },
    'amp': USE_AMP, 'channels_last': USE_CHANNELS_LAST,
    'fp32_transposed_conv': FP32_TRANSPOSED_CONV,
    'fused_adam': USE_FUSED_ADAM, 'cache_train_uint8': CACHE_RESIZED_IN_RAM,
    'drop_os_page_cache': DROP_OS_PAGE_CACHE, 'save_last_every': SAVE_LAST_EVERY,
    'prediction_threshold': PRED_THRESHOLD, 'jaccard_cut': JACCARD_CUT,
    'protocol_version': PROTOCOL_VERSION,
}
(RUN_DIR / 'config.json').write_text(
    json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False), encoding='utf-8'
)

def train_fold(fold_number, train_subset, holdout_subset):
    fold_seed = SEED + fold_number - 1
    set_seed(fold_seed)
    first_marker = CHECKPOINT_DIR / f'fold_{fold_number}' / f'{SIZES[0]}px_frozen' / 'last.pt'
    continuing = RESUME and first_marker.is_file()
    model = to_runtime(UNet34(pretrained=PRETRAINED and not continuing))
    stage_results = []
    for size_index, size in enumerate(SIZES):
        freeze_first_layer_group(model)
        stage_results.append(run_stage(
            model, train_subset, holdout_subset, size, FROZEN_EPOCHS, False,
            fold_number, 'frozen', fold_seed + size_index * 1000,
        ))
        unfreeze_with_frozen_bn(model)
        stage_results.append(run_stage(
            model, train_subset, holdout_subset, size, UNFROZEN_EPOCHS, True,
            fold_number, 'unfrozen', fold_seed + size_index * 1000 + 500,
        ))

    holdout_loader = make_loader(holdout_subset, SIZES[-1], train=False, seed=fold_seed)
    final_holdout_dice = evaluate_dice(model, holdout_loader)
    final_path = CHECKPOINT_DIR / f'fold_{fold_number}' / 'unet34_best.pt'
    metadata = {
        'fold': fold_number, 'fold_seed': fold_seed, 'config': RUN_CONFIG,
        'train_count': len(train_subset), 'holdout_count': len(holdout_subset),
        'train_ids': ids_digest(train_subset), 'holdout_ids': ids_digest(holdout_subset),
        'holdout_dice': final_holdout_dice, 'stage_results': stage_results,
    }
    atomic_torch_save({'model_state': unwrap_model(model).state_dict(),
                       'meta': metadata}, final_path)
    rows = [row for result in stage_results for row in result['history']]
    pd.DataFrame(rows).to_csv(LOG_DIR / f'fold_{fold_number}_history.csv', index=False)
    del model, holdout_loader
    gc.collect()
    torch.cuda.empty_cache()
    return final_path, metadata

fold_checkpoint_paths, fold_metadata = [], []
training_started = time.time()
for fold_number, (train_subset, holdout_subset) in enumerate(folds, start=1):
    final_path = CHECKPOINT_DIR / f'fold_{fold_number}' / 'unet34_best.pt'
    if RESUME and final_path.is_file():
        payload = trusted_torch_load(final_path, map_location='cpu')
        metadata = payload['meta']
        if (metadata.get('config') != RUN_CONFIG
                or metadata.get('train_ids') != ids_digest(train_subset)
                or metadata.get('holdout_ids') != ids_digest(holdout_subset)):
            raise RuntimeError(f'Fold checkpoint không khớp: {final_path}')
        del payload
        fold_path = final_path
        print(f"[resume] fold {fold_number} complete, Dice={metadata['holdout_dice']:.4f}")
    else:
        print(f'\n===== FOLD {fold_number}/{N_FOLDS}: train={len(train_subset)}, '
              f'holdout={len(holdout_subset)} =====')
        fold_path, metadata = train_fold(fold_number, train_subset, holdout_subset)
    fold_checkpoint_paths.append(str(fold_path))
    fold_metadata.append(metadata)

fold_table = pd.DataFrame([
    {'fold': metadata['fold'], 'train_count': metadata['train_count'],
     'holdout_count': metadata['holdout_count'],
     'internal_holdout_dice': metadata['holdout_dice']}
    for metadata in fold_metadata
])
training_elapsed_hours = (time.time() - training_started) / 3600
fold_table.to_csv(LOG_DIR / 'fold_holdout_dice.csv', index=False)
display(fold_table.style.format({'internal_holdout_dice': '{:.4f}'}))
print(f'Training complete in {training_elapsed_hours:.2f} hours. '
      'Starting official validation-only evaluation in the next cell.')
print('Checkpoints:', fold_checkpoint_paths)

## Chỉ đánh giá official validation

Official validation chỉ dùng để báo cáo từng fold và equal-weight probability ensemble, không tham gia train hoặc chọn checkpoint. Notebook đồng thời xếp hạng từng ảnh validation bằng Dice của ensemble để in Top 10 tốt nhất và Top 10 tệ nhất, kèm visualization input/ground truth/prediction/diff. HD95 có đơn vị pixel trên mask đã resize `256×256`.

In [ ]:
# Keep all fold models resident on cuda:0. T4 has enough VRAM for these U-Net34
# checkpoints; each model is forwarded once per batch and reused for ensemble metrics.
runtime_models = []
for checkpoint_path in fold_checkpoint_paths:
    payload = trusted_torch_load(checkpoint_path, map_location='cpu')
    model = to_runtime(UNet34(pretrained=False))
    load_model_state(model, payload['model_state'])
    model.eval()
    runtime_models.append(model)
    del payload

gc.collect()
torch.cuda.empty_cache()

EVALUATION_CONFIG = {
    'device': str(DEVICE),
    'gpu': GPU_NAMES[0],
    'batch_size': EVAL_BATCH_SIZE,
    'amp_fp16_inference': USE_AMP,
    'metrics_backend': 'PyTorch CUDA',
    'hd95_backend': 'torch.cdist CUDA surface distance',
    'hd95_source_chunk': HD95_SOURCE_CHUNK,
    'hd95_target_chunk': HD95_TARGET_CHUNK,
    'single_pass_reuse_for_ensemble': True,
}
print('Evaluation config:', EVALUATION_CONFIG)

# -------- Official Validation: all folds + ensemble, ONE pass through the loader --------
validation_loader = make_loader(
    official_val_records,
    SIZES[-1],
    train=False,
    seed=SEED,
    batch_size=EVAL_BATCH_SIZE,
)
validation_started = time.perf_counter()
validation_results, validation_per_image_rows = evaluate_variants_gpu(
    runtime_models,
    validation_loader,
    individual_indices=range(len(runtime_models)),
    include_ensemble=True,
    include_hd95=True,
    sample_ids=[record['image_id'] for record in official_val_records],
)
torch.cuda.synchronize()
validation_seconds = time.perf_counter() - validation_started

validation_rows = []
for index in range(len(runtime_models)):
    validation_rows.append({
        'model': f'BestDice fold {index + 1}',
        **validation_results[f'model_{index}'],
    })
validation_rows.append({
    'model': 'Equal-weight ensemble',
    **validation_results['ensemble'],
})
validation_table = pd.DataFrame(validation_rows)
validation_table.to_csv(LOG_DIR / 'official_validation_metrics.csv', index=False)

validation_per_image_table = pd.DataFrame(validation_per_image_rows)
if len(validation_per_image_table) != len(official_val_records):
    raise RuntimeError('Thiếu kết quả per-image trên official validation')
validation_per_image_table = validation_per_image_table.sort_values(
    'image_id', kind='stable'
).reset_index(drop=True)
validation_per_image_table.to_csv(
    LOG_DIR / 'official_validation_per_image_ensemble.csv', index=False
)

ranking_columns = [
    'image_id', 'dice', 'jaccard', 'threshold_jaccard', 'sensitivity',
    'predicted_pixels', 'target_pixels',
]
top10_best = validation_per_image_table.sort_values(
    ['dice', 'jaccard', 'image_id'],
    ascending=[False, False, True], kind='stable',
).head(10)[ranking_columns].copy()
top10_worst = validation_per_image_table.sort_values(
    ['dice', 'jaccard', 'image_id'],
    ascending=[True, True, True], kind='stable',
).head(10)[ranking_columns].copy()
top10_best.insert(0, 'rank', range(1, len(top10_best) + 1))
top10_worst.insert(0, 'rank', range(1, len(top10_worst) + 1))
top10_best.to_csv(LOG_DIR / 'official_validation_top10_best.csv', index=False)
top10_worst.to_csv(LOG_DIR / 'official_validation_top10_worst.csv', index=False)

@torch.inference_mode()
def predict_ensemble_mask(image_rgb, models_list):
    if not models_list:
        raise ValueError('models_list rỗng')
    image_tensor = torch.from_numpy(
        np.ascontiguousarray(image_rgb)
    ).permute(2, 0, 1).unsqueeze(0)
    inputs = prepare_inputs(image_tensor)
    ensemble_probability = torch.zeros(
        (1, 1, image_rgb.shape[0], image_rgb.shape[1]),
        device=DEVICE, dtype=torch.float32,
    )
    for model in models_list:
        ensemble_probability.add_(model_probability_gpu(model, inputs))
    ensemble_probability.div_(len(models_list))
    prediction = (
        ensemble_probability[0, 0] > PRED_THRESHOLD
    ).detach().cpu().numpy()
    return prediction

def segmentation_diff_rgb(target, prediction):
    target = target.astype(bool, copy=False)
    prediction = prediction.astype(bool, copy=False)
    diff = np.zeros((*target.shape, 3), dtype=np.float32)
    diff[target & prediction] = (0.10, 0.80, 0.20)       # TP: green
    diff[~target & prediction] = (1.00, 0.10, 0.10)      # FP: red
    diff[target & ~prediction] = (0.10, 0.45, 1.00)      # FN: blue
    return diff

def visualize_ranked_predictions(ranking_table, title, output_path):
    if ranking_table.empty:
        raise ValueError('ranking_table rỗng')
    records_by_id = {
        record['image_id']: record for record in official_val_records
    }
    cases_per_row = 2
    rows = math.ceil(len(ranking_table) / cases_per_row)
    figure, axes = plt.subplots(rows, 8, figsize=(24, 4.2 * rows))
    axes = np.asarray(axes, dtype=object).reshape(rows, 8)
    for axis in axes.ravel():
        axis.axis('off')

    for case_index, (_, ranked) in enumerate(ranking_table.iterrows()):
        image_id = str(ranked['image_id'])
        if image_id not in records_by_id:
            raise KeyError(f'Không tìm thấy validation record: {image_id}')
        record = records_by_id[image_id]
        image_rgb = load_color_balanced_rgb(record['image_path'])
        image_rgb = cv2.resize(
            image_rgb, (SIZES[-1], SIZES[-1]), interpolation=cv2.INTER_LINEAR
        )
        target = load_pil_uint8(record['mask_path'], 'L')
        target = cv2.resize(
            target, (SIZES[-1], SIZES[-1]), interpolation=cv2.INTER_NEAREST
        ) > 127
        prediction = predict_ensemble_mask(image_rgb, runtime_models)
        diff = segmentation_diff_rgb(target, prediction)

        plot_row = case_index // cases_per_row
        first_column = (case_index % cases_per_row) * 4
        case_axes = axes[plot_row, first_column:first_column + 4]
        case_axes[0].imshow(image_rgb)
        case_axes[0].set_title(
            f"#{int(ranked['rank'])} {image_id}\n"
            f"Dice={ranked['dice']:.4f} | IoU={ranked['jaccard']:.4f}",
            fontsize=10,
        )
        case_axes[1].imshow(target, cmap='gray', vmin=0, vmax=1)
        case_axes[1].set_title('Ground truth', fontsize=10)
        case_axes[2].imshow(prediction, cmap='gray', vmin=0, vmax=1)
        case_axes[2].set_title('Prediction', fontsize=10)
        case_axes[3].imshow(diff, vmin=0, vmax=1)
        case_axes[3].set_title('Diff', fontsize=10)

    legend_handles = [
        Patch(facecolor=(0.10, 0.80, 0.20), label='TP: đúng foreground'),
        Patch(facecolor=(1.00, 0.10, 0.10), label='FP: dự đoán thừa'),
        Patch(facecolor=(0.10, 0.45, 1.00), label='FN: bỏ sót lesion'),
        Patch(facecolor=(0.00, 0.00, 0.00), label='TN: đúng background'),
    ]
    figure.suptitle(title, fontsize=16, fontweight='bold')
    figure.legend(
        handles=legend_handles, loc='lower center', ncol=4, frameon=False
    )
    figure.tight_layout(rect=(0, 0.035, 1, 0.965))
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(figure)
    return str(output_path)

top10_best_figure = visualize_ranked_predictions(
    top10_best,
    'TOP 10 OFFICIAL VALIDATION TỐT NHẤT — Equal-weight ensemble',
    LOG_DIR / 'official_validation_top10_best_predictions.png',
)
top10_worst_figure = visualize_ranked_predictions(
    top10_worst,
    'TOP 10 OFFICIAL VALIDATION TỆ NHẤT — Equal-weight ensemble',
    LOG_DIR / 'official_validation_top10_worst_predictions.png',
)
del validation_loader, validation_results, validation_per_image_rows

print(
    f'Official validation GPU evaluation: {validation_seconds:.2f}s | '
    f'{len(official_val_records) / validation_seconds:.1f} images/s | {memory_status()}'
)

summary = {
    'config': RUN_CONFIG,
    'evaluation_config': EVALUATION_CONFIG,
    'sample_counts': {
        'train': len(train_records),
        'validation': len(official_val_records),
    },
    'elapsed_training_hours': training_elapsed_hours,
    'evaluation_seconds': {'validation': validation_seconds},
    'fold_checkpoints': fold_checkpoint_paths,
    'fold_internal_holdout': fold_table.to_dict(orient='records'),
    'official_validation': validation_table.to_dict(orient='records'),
    'per_image_ranking_model': 'equal-weight ensemble',
    'per_image_ranking_metric': 'dice descending/ascending',
    'per_image_metrics_csv': str(
        LOG_DIR / 'official_validation_per_image_ensemble.csv'
    ),
    'top10_best': top10_best.to_dict(orient='records'),
    'top10_worst': top10_worst.to_dict(orient='records'),
    'top10_visualizations': {
        'best': top10_best_figure, 'worst': top10_worst_figure,
        'diff_legend': {
            'green': 'true_positive', 'red': 'false_positive',
            'blue': 'false_negative', 'black': 'true_negative',
        },
    },
    'hd95_unit': f'pixel at {SIZES[-1]}x{SIZES[-1]} resolution',
}
summary_path = RUN_DIR / 'summary.json'
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

metric_format = {
    column: '{:.4f}'
    for column in validation_table.columns
    if column != 'model'
}
display(validation_table.style.format(metric_format))
ranking_format = {
    'dice': '{:.4f}', 'jaccard': '{:.4f}',
    'threshold_jaccard': '{:.4f}', 'sensitivity': '{:.4f}',
}
print('\nTOP 10 VALIDATION TỐT NHẤT — xếp theo ensemble Dice')
display(top10_best.style.format(ranking_format).hide(axis='index'))
print('\nTOP 10 VALIDATION TỆ NHẤT — xếp theo ensemble Dice')
display(top10_worst.style.format(ranking_format).hide(axis='index'))
del runtime_models
gc.collect()
torch.cuda.empty_cache()
print('Summary:', summary_path)


## Kết quả được tạo

- Ba checkpoint fold-final để resume và tái tạo equal-weight ensemble.
- Lịch sử train từng fold, bảng internal holdout và aggregate official-validation metrics trong `logs`.
- `official_validation_per_image_ensemble.csv` chứa metric ensemble của toàn bộ ảnh validation.
- `official_validation_top10_best.csv` và `official_validation_top10_worst.csv` chứa hai bảng xếp hạng theo Dice.
- Hai file PNG `official_validation_top10_*_predictions.png` hiển thị input, ground truth, ensemble prediction và diff màu cho Top 10 tốt/xấu.
- `summary.json` chứa cấu hình, split counts, checkpoint paths, aggregate metrics và Top 10.
- Official validation không tham gia LR Finder, augmentation, early stopping, chọn checkpoint, threshold hoặc ensemble weight. Official test không được nạp hay đánh giá.